In [1]:
from preamble import *

2025-06-05 16:47:25,648 - WARNING - File .dacerc not found. You are requesting data in public mode. To change this behaviour, create a .dacerc file in your home directory and fill it with your API key. More infos on https://dace.unige.ch


In [9]:
default_params={
    # Fixed parameters
    "log_g":4.52,
    "P_orb":8.463,
    "stellar_i":85.0,
    'P_rot':4.86,
    "Rs":0.8,
    "Ms":0.6,
    "Mp":10.0,
    "metallicity":0.12,
    "planet_i":89.5,
    "HST_period":0.066,
    # Ramp Model Parameters
    "r1": 18,
    "r2": -7,
    "r3": 0,
    #Breathing params
    "b1":0,
    "b2":0,
    "b3":0,
    "b4":0,
    # Transit Parameters
    "u1":0.22,
    "u2":0.35,
    "t0":0.0,
    "R0": 0.045,
    "a_rstar": 18.2,
    # "T_ref":600,
    "ecc": 0.045,
    "log_fixedspot_radii":-1.6,
    # "f_spot":0.2,
    # "spec_scale_factor": 1.0,
    "T_unocculted": 3100,
    "T_occulted": 3400,
    "T_phot": 4200,
    "spot1_lon":0.875,
    "spot1_lat":1.79,
    "spot1_rad":0.07,
    "spot2_lon":0.09,
    "spot2_lat":1.48,
    "spot2_rad":0.32,
    "spot3_lon":-1.1,
    "spot3_lat":1.2,
    "spot3_rad":0.1,
}

In [10]:
@jit
def spectroscopic_transit_model(parameters, stellar_obj):

    params = parameters
    active_star = stellar_obj
    # Map parameters to names
    # params = {name: value for name, value in zip(params_config.keys(), parameters)}

    # Calculate systematics model
    ramp = ramp_model(phase=ramp_phase_jax,
                    r1=params.get('r1', default_params['r1']),
                    r2=params.get('r2', default_params['r2']),
                    r3=params.get('r3', default_params['r3']))
    breathing = breathing_model(phase=breathing_phase_jax,
                              b1=params.get('b1', default_params['b1']),
                              b2=params.get('b2', default_params['b2']),
                              b3=params.get('b3', default_params['b3']),
                              b4=params.get('b4', default_params['b4']))
    systematics = (breathing * ramp)

    # Retrieve and bin model spectra
    _cool_occulted = get_BTSettl_spectrum_jax(T=jnp.array(params.get('T_occulted', default_params['T_occulted'])))
    _cool_unocculted = get_BTSettl_spectrum_jax(T=jnp.array(params.get('T_unocculted', default_params['T_unocculted'])))
    _phot = get_BTSettl_spectrum_jax(T=jnp.array(params.get('T_phot', default_params['T_phot'])))
    Cool_occulted = bin_spectrum(output_wavelength = data_wavelengths_jax,
                        input_wavelength = _cool_occulted[0],
                        input_flux = _cool_occulted[1])
    Cool_unocculted = bin_spectrum(output_wavelength = data_wavelengths_jax,
                        input_wavelength = _cool_unocculted[0],
                        input_flux = _cool_unocculted[1])
    Phot = bin_spectrum(output_wavelength = data_wavelengths_jax,
                        input_wavelength = _phot[0],
                        input_flux = _phot[1])

    active_star.spectrum = jnp.array([Cool_unocculted]*n_spots)
    active_star.temperature = jnp.array([params.get("T_unocculted", default_params["T_unocculted"])] * n_spots) 
    active_star.rad = jnp.array([10**(jnp.array(params['log_fixedspot_radii']))] * n_spots)
    # Now replace spots 0, 1, and 2 
    active_star.spectrum = active_star.spectrum.at[0].set(Cool_occulted)
    active_star.spectrum = active_star.spectrum.at[1].set(Cool_occulted)
    active_star.temperature = active_star.temperature.at[0].set(params.get("T_occulted", default_params["T_occulted"]))
    active_star.temperature = active_star.temperature.at[1].set(params.get("T_occulted", default_params["T_occulted"]))
    active_star.lon = active_star.lon.at[0].set(params.get(f"spot1_lon", default_params[f"spot1_lon"]) )
    active_star.lat = active_star.lat.at[0].set(params.get(f"spot1_lat", default_params[f"spot1_lat"]) )
    active_star.rad = active_star.rad.at[0].set(params.get(f"spot1_rad", default_params[f"spot1_rad"]) )
    active_star.lon = active_star.lon.at[1].set(params.get(f"spot2_lon", default_params[f"spot2_lon"]) )
    active_star.lat = active_star.lat.at[1].set(params.get(f"spot2_lat", default_params[f"spot2_lat"]) )
    active_star.rad = active_star.rad.at[1].set(params.get(f"spot2_rad", default_params[f"spot2_rad"]) )
    active_star.lon = active_star.lon.at[2].set(params.get(f"spot3_lon", default_params[f"spot3_lon"]) )
    active_star.lat = active_star.lat.at[2].set(params.get(f"spot3_lat", default_params[f"spot3_lat"]) )
    active_star.rad = active_star.rad.at[2].set(params.get(f"spot3_rad", default_params[f"spot3_rad"]) )

    #Update Planet Parameters
    planet_parameters = dict(
        period = default_params['P_orb'],
        inclination = jnp.radians(default_params['planet_i']),
        t0 = params.get('t0', default_params['t0']),
        # omega = np.radians(88.5),
        ecc = params.get('ecc', default_params['ecc']),
        a = params.get('a_rstar', default_params['a_rstar']),
        rp = jnp.array([params[f"rp_rs_w{j}"] for j in range(1,len(data_wavelengths_jax)+1)]),
        u1 = jnp.array([params[f"u1_w{j}"] for j in range(1,len(data_wavelengths_jax)+1)]),
        u2 = jnp.array([params[f"u2_w{j}"] for j in range(1,len(data_wavelengths_jax)+1)])
    )
    
    lc_model, contam, X, Y, spectrum_at_transit = active_star.transit_model(**planet_parameters)

    # Normalize each wavelength's light curve by its mean (time-axis normalization)
    means = jnp.mean(lc_model, axis=0)  # Shape: (n_wavelengths,)
    normalized_spectroscopic_model = (lc_model / means).T  # Transpose to (n_wavelengths, n_times)
    
    return normalized_spectroscopic_model

In [ ]:
def lnlike_OOTspec(parameters, plot=False, savefigs=False, samples=None,
                   convolution_method='astropy', kernel_type='astropy'):

    ln_like = 0.0

    'Map parameters to names for easier reference'
    params = {name: value for name, value in zip(params_config.keys(), parameters)}
        
    Cool = btsettl_grid(float(params['T_spot']))
    Phot = btsettl_grid(float(params['T_phot']))
    
    'Calculate combined spectrum'
    _f = (params['f_spot']*Cool + (1.0-params['f_spot'])*Phot)
    exclude_nans = ~np.isnan(_f)    
    model_wave = btsettl_wavelengths[exclude_nans]
    model_flux = _f[exclude_nans]
    convolved = convolve_spectrum(model_wave, model_flux, sigma=filter_sigma, method = convolution_method, kernel_type = kernel_type)
    binned_model_flux = bintogrid(model_wave, convolved, newx=mean_wave)['y'] * 3.44768e-18
    model_flux = binned_model_flux * params['spec_scale_factor']
    
    'Calculate ln_like for the light curve model'
    OOT_chisq = np.nansum(((calibrated_data_flux.value - model_flux) ** 2) / ((oot_spec_err*calibrated_data_flux.value) ** 2))
    OOT_err_weight = np.nansum(1.0 / np.sqrt(2.0 * np.pi * (oot_spec_err*calibrated_data_flux.value)))
    ln_like += (OOT_err_weight - 0.5 * OOT_chisq)

    if plot:
        plt.errorbar(mean_wave, calibrated_data_flux, yerr = (oot_spec_err*calibrated_data_flux),fmt='o',ms=2)
        plt.plot(mean_wave, model_flux)
        if savefigs:
            plt.savefig(f'../figs/{visit}_{n_spots}spots_{nsteps}steps_OOTSpec_trad.png',dpi=250)
        plt.show()
        plt.clf()

        dof = np.sum(~np.isnan(mean_wave)) - 4
        reduced_chisq = OOT_chisq/dof
        
        print(f'This Models Reduced Chisq = {reduced_chisq:.3f} for {4} parameters and {np.sum(~np.isnan(mean_wave))} data points')

    return ln_like

In [11]:
# def lnprob():

#     ln_prob = 0
#     ln_like = 0

#     if all(bounds[0] <= params[key] <= bounds[1] for key, bounds in priors.items()):

#         # Retrieve and bin model spectra
#         _cool_occulted = get_BTSettl_spectrum_jax(T=jnp.array(default_params['T_occulted']))
#         _cool_unocculted = get_BTSettl_spectrum_jax(T=jnp.array(default_params['T_unocculted']))
#         _phot = get_BTSettl_spectrum_jax(T=jnp.array(default_params['T_phot']))
#         Cool_occulted = bin_spectrum(output_wavelength = data_wavelengths_jax,
#                             input_wavelength = _cool_occulted[0],
#                             input_flux = _cool_occulted[1])
#         Cool_unocculted = bin_spectrum(output_wavelength = data_wavelengths_jax,
#                             input_wavelength = _cool_unocculted[0],
#                             input_flux = _cool_unocculted[1])
#         Phot = bin_spectrum(output_wavelength = data_wavelengths_jax,
#                             input_wavelength = _phot[0],
#                             input_flux = _phot[1])
#         # Make the Active Star
#         star = ActiveStar(
#             times=time_from_T0_jax,
#             inclination=jnp.radians(default_params['stellar_i']),
#             T_eff=default_params['T_phot'],
#             wavelength=data_wavelengths_jax/1e4,
#             phot=Phot,
#             P_rot=default_params['P_rot'])
#         star.spectrum = jnp.array([Cool_unocculted]*n_spots)
#         star.temperature = jnp.array([3100] * n_spots)
#         star.lon = jnp.array(initial_lons)
#         star.lat = jnp.array(initial_lats)    
#         star.rad = jnp.array([0.01]*n_spots)

#         ln_prob += ln_like

#     else ln_prob = -np.inf
    
#     return ln_prob

IndentationError: unexpected indent (2457369637.py, line 9)

In [12]:
n_spots = 800

F21_bin_edges = np.array([1.14064,1.1777,
                            1.21477,1.26573,
                            1.32132,1.35838,
                            1.45104,
                            1.53907,
                            1.60393,1.642])* u.micron
S22_bin_edges = np.array([0.84027,0.85502,0.87469,0.89436,
                            0.91402,0.93369,0.95336,0.97303,0.99269,
                            1.01236,1.03203,1.05170,1.07136,1.09103,
                            1.11070,1.13]) * u.micron

for visit in ['S22','F21']:

    print('Running ',visit)
    print('')

    # Read in and initialize the data
    spec_bin_edges = F21_bin_edges if visit=='F21' else S22_bin_edges
    predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
    exposureTime = visits[f'{visit}']['exp (s)'].value    
    rainbow = read_rainbow(f"../data/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
    binned_rainbow = rainbow.bin(wavelength_edges=spec_bin_edges)
    data_wavelength = binned_rainbow.wavelength.value
    img_date = binned_rainbow.time.value
    data_flux = binned_rainbow.flux.value
    relative_err = binned_rainbow.uncertainty.value
    time_from_T0 = img_date - predicted_T0
    for i in range(len(data_wavelength)):
            print(f'{visit} | Median relative uncertainty at {data_wavelength[i]:.4f} micron = {int(np.nanmedian(relative_err[i,:])*1e6)} ppm')

    # Label the orbits
    orbit = np.zeros_like(img_dates[0])
    for j in range(len(img_dates[0])):
        if j >= 1:
            if (img_dates[0][j] - img_dates[0][j - 1]) > 0.01:
                orbit[j] = (orbit[j - 1] + 1)
            else:
                orbit[j] = orbit[j - 1]
    
    # Trim the first point from each orbit
    ref_time = []
    for o in np.unique(orbit):
        first_index = np.where(orbit == o)[0][0]
        ref_time.append(img_dates[0][first_index])
        data_flux[:, first_index] = np.nan  # Set the first point of each subsequent orbit to np.nan
        relative_err[:, first_index] = np.nan
        img_date[first_index] = np.nan
        time_from_T0[first_index] = np.nan

    # Set data to nan if it was in the pre-defined list of orbits to exclude
    orbits_to_exclude = np.array([0]) if visit=='F21' else np.array([0,2])
    orbits_to_include = np.array([1,2,3,4,5,6,7]) if visit=='F21' else np.array([1,3,4,5,6,7])
    for orbit_to_exclude in orbits_to_exclude:
        data_flux[:, orbit == orbit_to_exclude] = np.nan
        relative_err[:, orbit == orbit_to_exclude] = np.nan
        img_date[orbit == orbit_to_exclude] = np.nan
        time_from_T0[orbit == orbit_to_exclude] = np.nan

    # Populate ramp_phase time arrays
    phase_list=[]
    for o in [0,1,2,3,4,5,6,7]:
        rphase = (img_date[orbit==o] - ref_time[o]) / 0.066
        phase_list.append(rphase)
    ramp_phase = np.concatenate(phase_list)
    breathing_phase = ( (img_date-ref_time[0]+0.02) / default_params['HST_period'] ) % 1

    # This nanmask makes the arrays safe for JAX
    nanmask = ~np.isnan(img_date)
    data_flux_filtered = data_flux[:,nanmask]
    relative_err_filtered = relative_err[:,nanmask]
    time_from_T0_filtered = time_from_T0[nanmask]  # assuming same length
    img_date_filtered = img_date[nanmask]          # assuming same length
    ramp_phase_filtered = ramp_phase[nanmask]
    breathing_phase_filtered = breathing_phase[nanmask]
    
    # Convert to JAX arrays
    _data_flux_jax = jnp.array(data_flux_filtered)
    means = jnp.mean(_data_flux_jax, axis=1, keepdims=True)
    data_flux_jax = _data_flux_jax / means
    relative_err_jax = jnp.array(relative_err_filtered)
    time_from_T0_jax = jnp.array(time_from_T0_filtered)
    img_date_jax = jnp.array(img_date_filtered)
    ramp_phase_jax = jnp.array(ramp_phase_filtered)
    data_wavelengths_jax = jnp.array(data_wavelength)
    breathing_phase_jax = jnp.array(breathing_phase_filtered)
    
    _cool_unocculted = get_BTSettl_spectrum_jax(T=jnp.array(3100) )
    _cool_occulted = get_BTSettl_spectrum_jax(T=jnp.array(3400) )
    _phot = get_BTSettl_spectrum_jax(T=jnp.array(4000) )
    Cool_unocculted = bin_spectrum(output_wavelength = data_wavelengths_jax,
                        input_wavelength = _cool_unocculted[0],
                        input_flux = _cool_unocculted[1])
    Cool_occulted = bin_spectrum(output_wavelength = data_wavelengths_jax,
                        input_wavelength = _cool_occulted[0],
                        input_flux = _cool_occulted[1])
    Phot = bin_spectrum(output_wavelength = data_wavelengths_jax,
                        input_wavelength = _phot[0],
                        input_flux = _phot[1])
    
    planet_parameters = dict(
        inclination = np.radians(default_params['planet_i']),
        a = default_params['a_rstar'],
        rp = default_params['R0'],
        period = default_params['P_orb'],
        t0 = default_params['t0'],
        # omega = np.radians(88.5),
        ecc = default_params['ecc'],
        u1 = default_params['u1'],
        u2 = default_params['u2']
    )
    
    active_star = ActiveStar(
        times = time_from_T0_jax,
        inclination=jnp.radians(default_params['stellar_i']),
        T_eff=default_params['T_phot'],
        wavelength=data_wavelengths_jax/1e4,
        phot=Phot,
        P_rot=default_params['P_rot']
    )
    
    for i in range(1, n_spots + 1):
        # Generate random values within the parameter space for each spot
        lon = np.random.uniform(-2.8,2.8)
        lat = random.choice( [np.random.beta(2, 1)*(0.9 - 0.1) + 0.1, np.random.beta(1,2)*(3.04 - 2.24) + 2.24 ] )
        rad = 0.01
    
        latspot = dict(
            lon=lon,  # Longitude [rad]
            lat=lat,  # Latitude [rad]
            rad=rad,  # Radius [R_star]
            spectrum=Cool_unocculted,  # Use provided spectrum
            temperature=default_params['T_unocculted'],  # Use provided temperature
        )
        # Add the spot to the active star
        active_star.add_spot(**latspot)
    
    initial_lats = active_star.lat
    initial_lons = active_star.lon
    initial_radii = active_star.rad

    # Now we set up the sampler!
    lc_model = spectroscopic_transit_model(default_params,active_star)

    for i,lc_i in lc_model:
        plt.plot(time_from_T0_jax, lc_i-0.001*i)

Running  S22



🌈🤖 It looks like you're trying to bin in wavelength for a
`Rainbow` object that might not be normalized. In the
current version of `chromatic`, binning before normalizing
might give inaccurate results if the typical uncertainty
varies strongly with wavelength.

Please consider normalizing first, for example with
`rainbow.normalize().bin(...)`
so that all uncertainties will effectively be relative,
and the inverse variance weighting used for binning
wavelengths together will give more reasonable answers.

If you really need to bin before normalizing, please submit
an Issue at github.com/zkbt/chromatic/, and we'll try to
prioritize implementing a statistically sound solution as
soon as possible!



  0%|          | 0/160 [00:00<?, ?it/s]

🌈🤖 It looks like you're trying to bin in wavelength for a
`Rainbow` object that might not be normalized. In the
current version of `chromatic`, binning before normalizing
might give inaccurate results if the typical uncertainty
varies strongly with wavelength.

Please consider normalizing first, for example with
`rainbow.normalize().bin(...)`
so that all uncertainties will effectively be relative,
and the inverse variance weighting used for binning
wavelengths together will give more reasonable answers.

If you really need to bin before normalizing, please submit
an Issue at github.com/zkbt/chromatic/, and we'll try to
prioritize implementing a statistically sound solution as
soon as possible!



  0%|          | 0/160 [00:00<?, ?it/s]

S22 | Median relative uncertainty at 0.8476 micron = 481 ppm
S22 | Median relative uncertainty at 0.8649 micron = 451 ppm
S22 | Median relative uncertainty at 0.8845 micron = 405 ppm
S22 | Median relative uncertainty at 0.9042 micron = 379 ppm
S22 | Median relative uncertainty at 0.9239 micron = 354 ppm
S22 | Median relative uncertainty at 0.9435 micron = 341 ppm
S22 | Median relative uncertainty at 0.9632 micron = 328 ppm
S22 | Median relative uncertainty at 0.9829 micron = 320 ppm
S22 | Median relative uncertainty at 1.0025 micron = 315 ppm
S22 | Median relative uncertainty at 1.0222 micron = 307 ppm
S22 | Median relative uncertainty at 1.0419 micron = 303 ppm
S22 | Median relative uncertainty at 1.0615 micron = 301 ppm
S22 | Median relative uncertainty at 1.0812 micron = 298 ppm
S22 | Median relative uncertainty at 1.1009 micron = 298 ppm
S22 | Median relative uncertainty at 1.1203 micron = 300 ppm


KeyError: 'rp_rs_w1'

In [ ]:
'THE FOLLOWING IS FOR SPECTRAL MODEL FLUX NORMALIZATION'
for visit in ['S22','F21']:
    params_config = _params_config
    priors = _priors
    params = default_params
    oot_err_inflation = 150.0
    
    'Calculate the average OOT spectrum'
    exptime = visits[f'{visit}']['exp (s)']
    binwidth = visits[f'{visit}']['native resolution']
    grism = visits[f'{visit}']['Grism']
    unbinned_wavelengths = []
    median_spectra = []
    relative_uncertainties = []
    for direction in ['Forward','Reverse']:
        trimmed_r = read_rainbow(f"../data/{visit}_{direction}_trimmed_pacman_spec.rainbow.npy")
        median_spectra.append(trimmed_r.get_median_spectrum())
        unbinned_wavelengths.append(trimmed_r.wavelength.value)
        summed_spec = jnp.nansum(trimmed_r.flux,axis=1)
        relative_uncertainties.append(jnp.sqrt(summed_spec)/summed_spec)
    meanOOTspec = jnp.nanmean(jnp.array(median_spectra),axis=0)
    mean_wave = jnp.nanmean(jnp.array(unbinned_wavelengths),axis=0)
    relative_err_OOT = oot_err_inflation * jnp.sqrt(relative_uncertainties[0]**2+relative_uncertainties[1]**2)/jnp.sqrt(2)
    e_per_s = meanOOTspec / exptime
    e_per_s_per_angstrom = e_per_s / binwidth
    _w, _s, _e = read_sensitivity_curve(grism=grism)
    binned_filter_response = bintogrid(_w.value, _s.value, newx=mean_wave)['y'] * u.cm**2 / u.erg
    calibrated_data_flux = e_per_s_per_angstrom / binned_filter_response
    # plt.figure()
    # plt.title(f'{visit} Data Flux (mean between the median-per-scan spectrum)')
    # plt.plot(mean_wave,calibrated_data_flux,color='k')
    
    def calculate_edges(arr):
        midpoints = (arr[:-1] + arr[1:]) / 2
        left_end = arr[0] - (arr[1] - arr[0])/2
        right_end = arr[-1] + (arr[-1] - arr[-2])/2
        return jnp.concatenate([jnp.array([left_end]), midpoints, jnp.array([right_end])])
    OOT_spec_bin_edges = calculate_edges(mean_wave)*u.micron
    # Set binning preferences and download model spectra
    kwargs = dict(
        bin_specification='edges',
        bins=OOT_spec_bin_edges,
        min=OOT_spec_bin_edges.min(),
        max=OOT_spec_bin_edges.max(), 
        log=False
    )
    Cool, Phot = [
        bin_spectrum(
            get_BTSettl_spectrum_jax(T=Temp), **kwargs
        )
        for Temp in [params.get('T_spot', default_params['T_spot']),
                    params.get('T_phot', default_params['T_phot'])]
    ]
    Cool = Cool.flux
    Phot = Phot.flux
    # plt.figure()
    # plt.title(f'{visit} BT-Settl models (pre-convolution)')
    # plt.plot(mean_wave,Cool,color='r')
    # plt.plot(mean_wave,Phot, color='blue')
    # plt.legend()

    fullspec_bin_edges = jnp.linspace(0.3, 4, 1000)*u.micron
    kwargs = dict(
        bin_specification='edges',
        bins=fullspec_bin_edges,
        min=fullspec_bin_edges.min(),
        max=fullspec_bin_edges.max(), 
        log=False
    )
    fullspec_Cool, fullspec_Phot = [
        bin_spectrum(
            get_BTSettl_spectrum_jax(T=Temp), **kwargs
        )
        for Temp in [params.get('T_spot', default_params['T_spot']),
                    params.get('T_phot', default_params['T_phot'])]
    ]
    fullspec_Cool=fullspec_Cool.flux # units of erg / (Angstrom s cm^2)
    fullspec_Phot=fullspec_Phot.flux
    fullspec_wavelengths = fullspec_bin_edges[:-1]+(jnp.diff(fullspec_bin_edges)/2)*u.micron

    # plt.figure()
    # plt.title(f'{visit} Broadband BT-Settl models')
    # plt.plot(fullspec_wavelengths,fullspec_Cool,color='r')
    # plt.plot(fullspec_wavelengths,fullspec_Phot, color='blue')

    delta_lambda = jnp.median(jnp.diff(fullspec_bin_edges)*1e4)*u.angstrom
    integrated_spot_spec = (jnp.sum(fullspec_Cool * delta_lambda))
    integrated_phot_spec = (jnp.sum(fullspec_Phot * delta_lambda))
    nf_cool = ( (c.sigma_sb * params.get('T_spot', default_params['T_spot'])**4)/integrated_spot_spec )
    nf_phot = ( (c.sigma_sb * params.get('T_spot', default_params['T_spot'])**4)/integrated_phot_spec )
    normed_Cool = (Cool * nf_cool)
    normed_Phot = (Phot * nf_phot)

    # plt.figure()
    # plt.title(f'{visit} Model Spectra Normalized to the SB Law')
    # plt.plot(mean_wave,normed_Cool, color='r')
    # plt.plot(mean_wave,normed_Phot, color='blue')

    filter_sigma = 0.9 if visit=='F21' else 0.85

    # Calculate combined spectrum
    _f = (0.3*normed_Cool + 0.7*normed_Phot)
    # _f.value[0] = np.nan
    # _f.value[1] = np.nan
    # _f.value[-1] = np.nan
    print(_f)
    # exclude_nans = ~jnp.isnan(_f)    
    # model_wave = mean_wave[exclude_nans]
    # _model_flux = _f[exclude_nans]
    model_wave = mean_wave

    convolved_flux = convolve_spectrum_jax(model_wave, _f, sigma=filter_sigma)
    convolved_flux.at([0]).set(jnp.nan)
    model_flux = convolved_flux * 3.44768e-18 * 0.2
    
    # For gradients (e.g., in optimization)
    # grad_fn = jax.grad(lambda s: jnp.sum(convolve_spectrum_jax(model_wave, _model_flux, s)))
    # print('Grad',grad_fn(filter_sigma))  # Gradient w.r.t. sigma
    plt.figure()
    plt.title(f'{visit} Convolved Model Flux')
    plt.plot(model_wave, model_flux)
    plt.errorbar(mean_wave, calibrated_data_flux,yerr=relative_err_OOT*calibrated_data_flux,fmt='o',ms=1)